# Process GPI TDM H+T by City

Computes city-level median household income and total households from ACS block-group
data, then uses median income to normalise H+T (Housing + Transportation) costs into an
income-share metric.  `HPLUST{YYYY}` = `(HTCOST{YYYY} × 12) / MEDINC{YYYY}` (0–1
annual share).  `HTCOST{YYYY}` is the monthly combined H+T cost.  `MEDINC{YYYY}`,
`TOTPOP{YYYY}`, and `TOTHH{YYYY}` (total households) are also exported.

Block-group ACS income suppressed by the Census Bureau is filled via a hierarchical
fallback — BG → parent census tract → parent county — before city-level aggregation.
Tract and county reference data are cached locally as CSV files and only re-downloaded
when the file does not already exist in `.\Inputs\`.

**To force a full re-download** of tract or county ACS data (e.g. after adding a new
target year), delete the corresponding CSV from `.\Inputs\` before running.

**Workflow**
1. Setup — imports, `fill_acs_panel` helper, config, output directories
2. Load Inputs — city boundaries, H+T CSV, derive target years
3. Build 2020 Block Group Reference — fetch via pygris, persist to GDB (skipped if exists)
4. Fetch ACS Block Group Data by Year — `get_census` per year/county (income, pop, HH)
5. Estimate Missing BG ACS Years — interpolate gaps; extrapolate edges via OLS trend
6. Fetch & Cache Tract-Level ACS Fallback — download once to CSV; apply same interp/extrap
7. Fetch & Cache County-Level ACS Fallback — same for county level
8. Apply Hierarchical Income Fallback (BG → Tract → County) — fill suppressed BG income
9. Assign Block Groups to Cities — arcpy centroid spatial join (`HAVE_THEIR_CENTER_IN`)
10. Aggregate to City-Level Income — population-weighted median income; sum of households
11. Fetch & Cache ACS Place Income / Build Hybrid MEDINC — Place > County > BG-weighted
12. Compute H+T Income Share and Build Export — HTCOST, HPLUST, MEDINC, TOTPOP, TOTHH
13. Validate Export

## 1. Setup

In [188]:
import arcpy
from arcpy import env
import os
import re
import numpy as np
import pandas as pd
from arcgis import GIS
from arcgis.features import GeoAccessor, GeoSeriesAccessor
import geopandas as gpd
from pygris import block_groups
from pygris.data import get_census
from shapely.geometry import MultiPolygon, Polygon

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"
pd.options.display.max_columns = None


In [189]:
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")

if not CENSUS_API_KEY:
    raise ValueError("CENSUS_API_KEY is not set in the ArcGIS Pro Python environment.")


In [190]:
# Output directories
outputs = [".\\Outputs", "scratch.gdb", "Affordability_Housing_Transportation_Costs.gdb"]

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])


In [191]:
# Config for the city-level ACS / block-group / income workflow.
# All sections in this notebook read from this dictionary.
HT_INCOME_CONFIG = {
    "ht_csv_path": r".\Inputs\H +T Costs for Dashboard 2019-2023 (GPI & TDM) - Composite H + T Metric.csv",
    # ── Block-group spatial reference (geometry, fetched once via pygris) ──────
    "bg_reference_gdb": r".\Inputs\block_groups_2020.gdb",
    "bg_reference_name": "bg_2020_ut_5county",
    "bg_projected_name": "bg_2020_projected",
    "bg_city_join_name": "bg_city_spatial_join",
    # ── Tract- & county-level ACS fallback (tabular, cached to CSV) ───────────
    # Naming mirrors block_groups_2020 so the geography level is immediately clear.
    # Delete these files to force a full re-download (e.g. after adding a new year).
    "tract_ref_path": r".\Inputs\tracts_2020_acs_income.csv",
    "county_ref_path": r".\Inputs\counties_2020_acs_income.csv",
    "places_ref_path": r".\Inputs\places_2020_acs_income.csv",  # ACS Place income cache
    # ── Geography identifiers ─────────────────────────────────────────────────
    "state": "UT",
    "state_fips": "49",
    "county_fips": ["003", "011", "035", "049", "057"],
    "county_names": ["BOX ELDER", "DAVIS", "SALT LAKE", "UTAH", "WEBER"],
    # ── ACS variables fetched at every geographic level ───────────────────────
    "acs_dataset": "acs/acs5",
    "acs_vars": {
        "median_income": "B19013_001E",  # Median household income
        "population": "B02001_001E",  # Total population (weight for income avg)
        "households": "B11001_001E",  # Total households (weight for H+T cost rollup)
    },
    # ── Column / key names ────────────────────────────────────────────────────
    "bg_key": "GEOID",
    "tract_key": "tract_geoid",  # derived as bg_geoid[:11]
    "county_key": "county_geoid",  # derived as bg_geoid[:5]
    # ── Notebook behaviour ────────────────────────────────────────────────────
    "year_prefixes": ["HCOST", "TCOST", "HPLUST"],
    "target_years_override": None,
    "pygris_cache": True,
    "share_scale": "0-1",
    "city_key": "CITYAREA",
    "workshop_area_lookup": {
        "Box Elder (WFRC)": "Box Elder Wfrc",
        "North Davis County": "Davis County North",
        "South Davis County": "Davis County South",
        "North Salt Lake County": "Salt Lake County North",
        "Southwest Salt Lake County": "Salt Lake County Sw",
        "Southeast Salt Lake County": "Salt Lake County Se",
        "North Weber County": "Weber County North",
        "South Weber County": "Weber County South",
        "Central Utah County": "Utah County Central",
        "North Utah County": "Utah County North",
        "South Utah County": "Utah County South",
    },
}


In [192]:
def fill_acs_panel(panel_df, geoid_col, value_source_pairs, year_col="year"):
    """
    Apply linear interpolation (internal gaps) and OLS extrapolation (edge gaps)
    to a long-format ACS panel, processing each GEOID independently.

    Used for block-group, tract, and county panels — defined once here so the
    logic is not duplicated across Sections 5, 6, and 7.

    Parameters
    ----------
    panel_df : pd.DataFrame
        Long-format panel: one row per (geoid, year).
    geoid_col : str
        Column name of the geographic identifier.
    value_source_pairs : list of (str, str)
        Each tuple is (value_column, source_column).  The source column is
        updated with 'interpolated' or 'extrapolated' for filled rows.
    year_col : str
        Column name of the year integer (default 'year').

    Returns
    -------
    pd.DataFrame — same shape as input, gaps filled, source columns updated.
    """
    filled_parts = []

    for geoid, group in panel_df.groupby(geoid_col):
        group = group.sort_values(year_col).copy().reset_index(drop=True)

        for value_col, source_col in value_source_pairs:
            observed = group[[year_col, value_col]].dropna()
            original_null = group[value_col].isna().copy()

            if len(observed) >= 2:
                years_obs = observed[year_col].to_numpy(dtype=float)
                vals_obs = observed[value_col].to_numpy(dtype=float)
                first_year = int(years_obs[0])
                last_year = int(years_obs[-1])

                # 1. Fill internal gaps by linear interpolation.
                group[value_col] = group[value_col].interpolate(
                    method="linear", limit_area="inside"
                )

                # OLS trend over all observed years — used for edge extrapolation.
                slope, intercept = np.polyfit(years_obs, vals_obs, 1)

                # 2. Left-edge extrapolation.
                left_mask = group[value_col].isna() & (group[year_col] < first_year)
                if left_mask.any():
                    group.loc[left_mask, value_col] = (
                        slope * group.loc[left_mask, year_col] + intercept
                    )

                # 3. Right-edge extrapolation.
                right_mask = group[value_col].isna() & (group[year_col] > last_year)
                if right_mask.any():
                    group.loc[right_mask, value_col] = (
                        slope * group.loc[right_mask, year_col] + intercept
                    )

                # 4. Tag provenance for rows that were originally null but now filled.
                for idx in group.index:
                    if original_null.iloc[idx] and pd.notna(group.loc[idx, value_col]):
                        yr = group.loc[idx, year_col]
                        group.loc[idx, source_col] = (
                            "interpolated" if first_year < yr < last_year else "extrapolated"
                        )

            # Sign guards and rounding (applied regardless of whether fill ran).
            group[value_col] = group[value_col].round(0)
            group.loc[group[value_col] <= 0, value_col] = np.nan

        filled_parts.append(group)

    return pd.concat(filled_parts, ignore_index=True)


## 2. Load Inputs

In [193]:
# City boundary SEDF — used for export geometry and the spatial join.
# CITY_NAME is added as CITYAREA immediately so the join key is consistent throughout.
city_area_shp = pd.DataFrame.spatial.from_featureclass(
    r".\Inputs\city_area_with_workshop_areas.shp"
)

if (
    HT_INCOME_CONFIG["city_key"] not in city_area_shp.columns
    and "CITY_NAME" in city_area_shp.columns
):
    city_area_shp[HT_INCOME_CONFIG["city_key"]] = city_area_shp["CITY_NAME"]

print("City area rows:", len(city_area_shp))
city_area_shp.head()


City area rows: 109


,FID,CITY_NAME,SUBAREA,CO_NAME,SHAPE,CITYAREA
0,0,Alpine,North Utah County,UTAH,"{""rings"": [[[433081.68319999985, 4477344.8058]...",Alpine
1,1,Alta,NA,SALT LAKE,"{""rings"": [[[449359.0999999996, 4492074], [449...",Alta
2,2,American Fork,North Utah County,UTAH,"{""rings"": [[[433815.22979999986, 4466632.21309...",American Fork
3,3,Balance of BOX ELDER,NA,BOX ELDER,"{""rings"": [[[417479, 4578625.4], [417425.5, 45...",Balance of BOX ELDER
4,4,Benjamin,South Utah County,UTAH,"{""rings"": [[[438601.23319999967, 4437471.4559]...",Benjamin


In [194]:
# H+T cost CSV.  Replace -1 sentinel with 0 (no data for that city-year).
# 'City Area' is added as CITYAREA here; 'County' is dropped.
ht_df = pd.read_csv(HT_INCOME_CONFIG["ht_csv_path"])
ht_df = ht_df.replace(-1, 0)

if HT_INCOME_CONFIG["city_key"] not in ht_df.columns and "City Area" in ht_df.columns:
    ht_df[HT_INCOME_CONFIG["city_key"]] = ht_df["City Area"]

if "City Area" in ht_df.columns:
    ht_df = ht_df.drop(columns=["City Area"])

if "County" in ht_df.columns:
    ht_df = ht_df.drop(columns=["County"])

print("H+T rows:", len(ht_df))
ht_df.head()


H+T rows: 101


,HCOST2019,HCOST2020,HCOST2021,HCOST2022,HCOST2023,HCOST2024,TCOST2019,TCOST2020,TCOST2021,TCOST2022,TCOST2023,TCOST2024,HPLUST2019,HPLUST2020,HPLUST2021,HPLUST2022,HPLUST2023,HPLUST2024,CITYAREA
0,1829.0,1858.0,2074.0,2640.0,2867.0,4341.0,762.0,608.0,701.0,716.0,738.0,749.0,2592.0,2466.0,2776.0,3356.0,3604.0,5090.0,Alpine
1,0.0,0.0,0.0,0.0,0.0,0.0,504.0,336.0,408.0,447.0,474.0,441.0,0.0,0.0,0.0,0.0,0.0,0.0,Alta
2,1521.0,1600.0,1788.0,2213.0,2400.0,3366.0,429.0,352.0,394.0,406.0,418.0,426.0,1950.0,1951.0,2182.0,2619.0,2819.0,3792.0,American Fork
3,0.0,0.0,0.0,0.0,0.0,0.0,1235.0,914.0,1014.0,1140.0,1340.0,1141.0,0.0,0.0,0.0,0.0,0.0,0.0,Balance of BOX ELDER
4,0.0,0.0,0.0,0.0,0.0,0.0,587.0,486.0,550.0,551.0,558.0,536.0,0.0,0.0,0.0,0.0,0.0,0.0,Benjamin


In [195]:
# Derive target years from HCOST / TCOST / HPLUST column suffixes.
target_years = sorted(
    {
        int(col[-4:])
        for col in ht_df.columns
        if col[-4:].isdigit()
        and any(col.startswith(prefix) for prefix in HT_INCOME_CONFIG["year_prefixes"])
    }
)

if HT_INCOME_CONFIG["target_years_override"] is not None:
    target_years = HT_INCOME_CONFIG["target_years_override"]

print("Detected H+T years:", target_years)
print("Using override    :", HT_INCOME_CONFIG["target_years_override"] is not None)


Detected H+T years: [2019, 2020, 2021, 2022, 2023, 2024]
Using override    : False


In [196]:
# Validation: confirm city names in CSV match city names in shapefile.
ht_cities = set(ht_df[HT_INCOME_CONFIG["city_key"]].dropna())
shp_col = (
    HT_INCOME_CONFIG["city_key"]
    if HT_INCOME_CONFIG["city_key"] in city_area_shp.columns
    else "CITY_NAME"
)
shape_cities = set(city_area_shp[shp_col].dropna())

in_csv_not_shp = sorted(ht_cities - shape_cities)
in_shp_not_csv = sorted(shape_cities - ht_cities)

print("In CSV but NOT shapefile (investigate if non-empty):", in_csv_not_shp)
print("In shapefile but NOT CSV (geometry-only rows, expected):", in_shp_not_csv)


In CSV but NOT shapefile (investigate if non-empty): []
In shapefile but NOT CSV (geometry-only rows, expected): ['Camp Williams', 'Davis County', 'Lake Mountain', 'SL County East Cyns', 'South Cedar Valley', 'South Goshen Valley', 'Utah Lake', 'West Mountain']


## 3. Build 2020 Block Group Reference

A single 2020 Census block group layer is used as the stable spatial reference for all
target years.  It is fetched once via `pygris`, normalised to `MultiPolygon` (required by
the OpenFileGDB Fiona driver), and persisted to a file GDB.  Subsequent runs skip the
fetch if the layer already exists.

In [197]:
bg_reference_fc = os.path.join(
    HT_INCOME_CONFIG["bg_reference_gdb"], HT_INCOME_CONFIG["bg_reference_name"]
)

if not arcpy.Exists(HT_INCOME_CONFIG["bg_reference_gdb"]):
    arcpy.CreateFileGDB_management(r".\Inputs", "block_groups_2020.gdb")

if not arcpy.Exists(bg_reference_fc):
    bg_2020_gdf = block_groups(
        state=HT_INCOME_CONFIG["state"], year=2020, cache=HT_INCOME_CONFIG["pygris_cache"]
    )
    bg_2020_gdf = bg_2020_gdf[bg_2020_gdf["COUNTYFP"].isin(HT_INCOME_CONFIG["county_fips"])].copy()
    bg_2020_gdf = bg_2020_gdf[
        ["GEOID", "STATEFP", "COUNTYFP", "TRACTCE", "BLKGRPCE", "geometry"]
    ].copy()

    # Normalise mixed Polygon/MultiPolygon so the OpenFileGDB driver has a consistent type.
    bg_2020_gdf["geometry"] = bg_2020_gdf["geometry"].apply(
        lambda g: MultiPolygon([g]) if isinstance(g, Polygon) else g
    )
    bg_2020_gdf.to_file(
        HT_INCOME_CONFIG["bg_reference_gdb"],
        layer=HT_INCOME_CONFIG["bg_reference_name"],
        driver="OpenFileGDB",
    )
    print("Exported:", bg_reference_fc)
    print("Rows    :", len(bg_2020_gdf))
    print("Counties:", sorted(bg_2020_gdf["COUNTYFP"].unique().tolist()))
    print("CRS     :", bg_2020_gdf.crs)
else:
    print("Using existing block group reference:", bg_reference_fc)


Using existing block group reference: .\Inputs\block_groups_2020.gdb\bg_2020_ut_5county


In [198]:
# Read back as arcgis SEDF.  Cast GEOID to str — all downstream joins depend on this.
block_groups_ref = pd.DataFrame.spatial.from_featureclass(bg_reference_fc)
block_groups_ref[HT_INCOME_CONFIG["bg_key"]] = block_groups_ref[HT_INCOME_CONFIG["bg_key"]].astype(
    str
)

print("Columns:", block_groups_ref.columns.tolist())
print("Shape  :", block_groups_ref.shape)
block_groups_ref.head()


Columns: ['OBJECTID', 'GEOID', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'BLKGRPCE', 'SHAPE']
Shape  : (1547, 7)


,OBJECTID,GEOID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,SHAPE
0,1,490351113061,49,035,111306,1,"{""rings"": [[[-111.83382099999994, 40.615462000..."
1,2,490351113062,49,035,111306,2,"{""rings"": [[[-111.82158699999997, 40.607759000..."
2,3,490351114001,49,035,111400,1,"{""rings"": [[[-111.88258799999994, 40.718417000..."
3,4,490351114002,49,035,111400,2,"{""rings"": [[[-111.88262299999997, 40.712825000..."
4,5,490351114003,49,035,111400,3,"{""rings"": [[[-111.88267499999995, 40.704964000..."


## 4. Fetch ACS Block Group Data by Year

For each target year, `get_census` retrieves median household income (`B19013_001E`),
total population (`B02001_001E`), and total households (`B11001_001E`) at the block-group
level across all 5 counties.  ACS data is left-joined onto the 2020 BG reference geometry.

Years where the API call fails are logged but do not stop execution — missing years are
handled in Section 5.

In [199]:
bg_acs_by_year = {}
acs_year_status = []

for year in target_years:
    county_frames = []
    year_ok = True
    year_message = "ok"

    for county_fips in HT_INCOME_CONFIG["county_fips"]:
        try:
            county_df = get_census(
                dataset=HT_INCOME_CONFIG["acs_dataset"],
                variables=list(HT_INCOME_CONFIG["acs_vars"].values()),
                year=year,
                params={
                    "for": "block group:*",
                    "in": (f"state:{HT_INCOME_CONFIG['state_fips']} county:{county_fips}"),
                    "key": CENSUS_API_KEY,
                },
                return_geoid=True,
                guess_dtypes=True,
            )
            county_frames.append(county_df)
        except Exception as exc:
            year_ok = False
            year_message = str(exc)
            break

    if year_ok and county_frames:
        acs_df = pd.concat(county_frames, ignore_index=True)
        acs_df = acs_df.rename(
            columns={
                HT_INCOME_CONFIG["acs_vars"]["median_income"]: "median_income",
                HT_INCOME_CONFIG["acs_vars"]["population"]: "population",
                HT_INCOME_CONFIG["acs_vars"]["households"]: "households",
            }
        )
        acs_df[HT_INCOME_CONFIG["bg_key"]] = acs_df[HT_INCOME_CONFIG["bg_key"]].astype(str)

        for col in ("median_income", "population", "households"):
            acs_df[col] = pd.to_numeric(acs_df[col], errors="coerce")
            acs_df.loc[acs_df[col] <= 0, col] = np.nan

        acs_df["acs_year"] = year
        acs_df["income_source"] = "acs"
        acs_df["population_source"] = "acs"
        acs_df["households_source"] = "acs"

        acs_df = acs_df[
            [
                HT_INCOME_CONFIG["bg_key"],
                "median_income",
                "population",
                "households",
                "acs_year",
                "income_source",
                "population_source",
                "households_source",
            ]
        ].copy()

        bg_acs_by_year[year] = block_groups_ref.merge(
            acs_df, on=HT_INCOME_CONFIG["bg_key"], how="left"
        )
        acs_year_status.append(
            {
                "year": year,
                "status": "fetched",
                "acs_rows": len(acs_df),
                "matched_inc_rows": bg_acs_by_year[year]["median_income"].notna().sum(),
                "matched_hh_rows": bg_acs_by_year[year]["households"].notna().sum(),
                "message": "ok",
            }
        )
    else:
        bg_acs_by_year[year] = block_groups_ref.copy()
        for col in ("median_income", "population", "households"):
            bg_acs_by_year[year][col] = np.nan
        bg_acs_by_year[year]["acs_year"] = year
        bg_acs_by_year[year]["income_source"] = np.nan
        bg_acs_by_year[year]["population_source"] = np.nan
        bg_acs_by_year[year]["households_source"] = np.nan
        acs_year_status.append(
            {
                "year": year,
                "status": "missing",
                "acs_rows": 0,
                "matched_inc_rows": 0,
                "matched_hh_rows": 0,
                "message": year_message,
            }
        )


In [200]:
# Fetch summary — check for any missing years before proceeding.
acs_year_status_df = pd.DataFrame(acs_year_status)
acs_year_status_df


,year,status,acs_rows,matched_inc_rows,matched_hh_rows,message
0,2019,fetched,1297,1053,1057,ok
1,2020,fetched,1547,1497,1530,ok
2,2021,fetched,1547,1503,1530,ok
3,2022,fetched,1547,1496,1532,ok
4,2023,fetched,1547,1496,1532,ok
5,2024,fetched,1547,1501,1535,ok


In [201]:
# Spot check: confirm ACS fields joined correctly for the first year.
sample_year = target_years[0]
print("Sample year:", sample_year)
print("Shape      :", bg_acs_by_year[sample_year].shape)
bg_acs_by_year[sample_year][
    [
        HT_INCOME_CONFIG["bg_key"],
        "median_income",
        "population",
        "households",
        "acs_year",
        "income_source",
    ]
].head()


Sample year: 2019
Shape      : (1547, 14)


,GEOID,median_income,population,households,acs_year,income_source
0,490351113061,75438.0,1792.0,723.0,2019.0,acs
1,490351113062,128194.0,839.0,335.0,2019.0,acs
2,490351114001,68387.0,1330.0,409.0,2019.0,acs
3,490351114002,78548.0,1086.0,396.0,2019.0,acs
4,490351114003,51789.0,1734.0,730.0,2019.0,acs


## 5. Estimate Missing BG ACS Years

Each block group is processed independently across all target years using the
`fill_acs_panel` helper defined in Section 1:

- **Internal gaps** (e.g. 2020 missing when 2019 and 2021 exist) are filled by linear
  interpolation between the two nearest flanking observed values.
- **Edge gaps** (leading or trailing years) are filled by linear extrapolation using an
  OLS trend fitted to **all** observed years for that block group.
- Block groups with fewer than 2 observed years are left as `NaN`.

Applies to `median_income`, `population`, and `households`.  Observed ACS values are
never overwritten.  Provenance is tracked via `income_source`, `population_source`, and
`households_source`: `"acs"` | `"interpolated"` | `"extrapolated"`.

In [202]:
# Build a long panel (one row per GEOID x year) to drive the fill logic.
acs_panel_parts = []
for year in target_years:
    year_df = bg_acs_by_year[year][
        [
            HT_INCOME_CONFIG["bg_key"],
            "median_income",
            "population",
            "households",
            "income_source",
            "population_source",
            "households_source",
        ]
    ].copy()
    year_df["year"] = year
    acs_panel_parts.append(year_df)

acs_panel_df = pd.concat(acs_panel_parts, ignore_index=True)
print(
    f"Panel shape: {acs_panel_df.shape}  ({len(target_years)} years x {len(block_groups_ref)} BGs)"
)
acs_panel_df.head()


Panel shape: (9282, 8)  (6 years x 1547 BGs)


,GEOID,median_income,population,households,income_source,population_source,households_source,year
0,490351113061,75438.0,1792.0,723.0,acs,acs,acs,2019
1,490351113062,128194.0,839.0,335.0,acs,acs,acs,2019
2,490351114001,68387.0,1330.0,409.0,acs,acs,acs,2019
3,490351114002,78548.0,1086.0,396.0,acs,acs,acs,2019
4,490351114003,51789.0,1734.0,730.0,acs,acs,acs,2019


In [203]:
# Run fill_acs_panel on the block-group panel.
# The helper handles all three variables in one pass — see Section 1 for the full docstring.
acs_panel_filled_df = fill_acs_panel(
    acs_panel_df,
    geoid_col=HT_INCOME_CONFIG["bg_key"],
    value_source_pairs=[
        ("median_income", "income_source"),
        ("population", "population_source"),
        ("households", "households_source"),
    ],
)
print("Filled panel shape:", acs_panel_filled_df.shape)
acs_panel_filled_df.head()


Filled panel shape: (9282, 8)


,GEOID,median_income,population,households,income_source,population_source,households_source,year
0,490039601001,65682.0,766.0,236.0,acs,acs,acs,2019
1,490039601001,69875.0,874.0,263.0,acs,acs,acs,2020
2,490039601001,82500.0,1348.0,360.0,acs,acs,acs,2021
3,490039601001,88542.0,1174.0,319.0,acs,acs,acs,2022
4,490039601001,92262.0,1150.0,328.0,acs,acs,acs,2023


In [204]:
# Fill provenance summary by year.
fill_summary_df = (
    acs_panel_filled_df.groupby("year")
    .agg(
        income_acs=pd.NamedAgg(column="income_source", aggfunc=lambda s: (s == "acs").sum()),
        income_interpolated=pd.NamedAgg(
            column="income_source", aggfunc=lambda s: (s == "interpolated").sum()
        ),
        income_extrapolated=pd.NamedAgg(
            column="income_source", aggfunc=lambda s: (s == "extrapolated").sum()
        ),
        pop_acs=pd.NamedAgg(column="population_source", aggfunc=lambda s: (s == "acs").sum()),
        pop_interpolated=pd.NamedAgg(
            column="population_source", aggfunc=lambda s: (s == "interpolated").sum()
        ),
        pop_extrapolated=pd.NamedAgg(
            column="population_source", aggfunc=lambda s: (s == "extrapolated").sum()
        ),
        hh_acs=pd.NamedAgg(column="households_source", aggfunc=lambda s: (s == "acs").sum()),
        hh_interpolated=pd.NamedAgg(
            column="households_source", aggfunc=lambda s: (s == "interpolated").sum()
        ),
        hh_extrapolated=pd.NamedAgg(
            column="households_source", aggfunc=lambda s: (s == "extrapolated").sum()
        ),
    )
    .reset_index()
)
fill_summary_df


,year,income_acs,income_interpolated,income_extrapolated,pop_acs,pop_interpolated,pop_extrapolated,hh_acs,hh_interpolated,hh_extrapolated
0,2019,1058,0,474,1063,0,474,1060,0,476
1,2020,1517,11,19,1543,0,4,1544,0,3
2,2021,1523,19,5,1545,0,2,1545,1,1
3,2022,1517,25,5,1547,0,0,1547,0,0
4,2023,1517,20,10,1547,0,0,1547,0,0
5,2024,1519,0,28,1547,0,0,1547,0,0


In [205]:
# Spot check: all years for one GEOID should have sensible values and sources.
sample_geoid = acs_panel_filled_df[HT_INCOME_CONFIG["bg_key"]].iloc[0]
acs_panel_filled_df[acs_panel_filled_df[HT_INCOME_CONFIG["bg_key"]] == sample_geoid].sort_values(
    "year"
)[["year", "median_income", "population", "households", "income_source", "households_source"]]


,year,median_income,population,households,income_source,households_source
0,2019,65682.0,766.0,236.0,acs,acs
1,2020,69875.0,874.0,263.0,acs,acs
2,2021,82500.0,1348.0,360.0,acs,acs
3,2022,88542.0,1174.0,319.0,acs,acs
4,2023,92262.0,1150.0,328.0,acs,acs
5,2024,101100.0,1173.0,342.0,acs,acs


In [206]:
# Write filled values back into bg_acs_by_year, replacing the raw ACS columns.
for year in target_years:
    year_fill = acs_panel_filled_df[acs_panel_filled_df["year"] == year][
        [
            HT_INCOME_CONFIG["bg_key"],
            "median_income",
            "population",
            "households",
            "income_source",
            "population_source",
            "households_source",
        ]
    ].copy()

    base_cols = [
        col
        for col in bg_acs_by_year[year].columns
        if col
        not in [
            "median_income",
            "population",
            "households",
            "income_source",
            "population_source",
            "households_source",
        ]
    ]
    bg_acs_by_year[year] = bg_acs_by_year[year][base_cols].merge(
        year_fill, on=HT_INCOME_CONFIG["bg_key"], how="left"
    )


## 6. Fetch & Cache Tract-Level ACS Fallback

Census tract data is virtually never suppressed for `B19013_001E` because tracts are
designed to contain 2,500–8,000 people — large enough for reliable ACS estimates.  This
section downloads tract-level income, population, and household counts for all 5 counties,
applies the same `fill_acs_panel` interpolation/extrapolation pipeline used for block
groups, and saves the filled panel to a CSV.

**File:** `.\Inputs\tracts_2020_acs_income.csv`  
**Skipped:** if the file already exists.  Delete to force a full re-download.

In [207]:
tract_ref_path = HT_INCOME_CONFIG["tract_ref_path"]

if os.path.exists(tract_ref_path):
    tract_acs_filled_df = pd.read_csv(tract_ref_path, dtype={HT_INCOME_CONFIG["tract_key"]: str})
    print("Loaded cached tract ACS data:", tract_ref_path)
    print("Shape:", tract_acs_filled_df.shape)
else:
    print("Fetching tract-level ACS data for all target years...")
    tract_raw_parts = []
    tract_year_status = []

    for year in target_years:
        county_frames = []
        year_ok = True
        year_message = "ok"

        for county_fips in HT_INCOME_CONFIG["county_fips"]:
            try:
                county_df = get_census(
                    dataset=HT_INCOME_CONFIG["acs_dataset"],
                    variables=list(HT_INCOME_CONFIG["acs_vars"].values()),
                    year=year,
                    params={
                        "for": "tract:*",
                        "in": (f"state:{HT_INCOME_CONFIG['state_fips']} county:{county_fips}"),
                        "key": CENSUS_API_KEY,
                    },
                    return_geoid=True,
                    guess_dtypes=True,
                )
                county_frames.append(county_df)
            except Exception as exc:
                year_ok = False
                year_message = str(exc)
                break

        if year_ok and county_frames:
            acs_df = pd.concat(county_frames, ignore_index=True)
            acs_df = acs_df.rename(
                columns={
                    HT_INCOME_CONFIG["acs_vars"]["median_income"]: "median_income",
                    HT_INCOME_CONFIG["acs_vars"]["population"]: "population",
                    HT_INCOME_CONFIG["acs_vars"]["households"]: "households",
                }
            )
            acs_df[HT_INCOME_CONFIG["tract_key"]] = acs_df[HT_INCOME_CONFIG["bg_key"]].astype(str)
            for col in ("median_income", "population", "households"):
                acs_df[col] = pd.to_numeric(acs_df[col], errors="coerce")
                acs_df.loc[acs_df[col] <= 0, col] = np.nan

            acs_df["year"] = year
            acs_df["income_source"] = "acs"
            acs_df["population_source"] = "acs"
            acs_df["households_source"] = "acs"

            tract_raw_parts.append(
                acs_df[
                    [
                        HT_INCOME_CONFIG["tract_key"],
                        "median_income",
                        "population",
                        "households",
                        "year",
                        "income_source",
                        "population_source",
                        "households_source",
                    ]
                ]
            )
            tract_year_status.append(
                {"year": year, "status": "fetched", "rows": len(acs_df), "message": "ok"}
            )
        else:
            tract_year_status.append(
                {"year": year, "status": "missing", "rows": 0, "message": year_message}
            )

    print(pd.DataFrame(tract_year_status))

    if not tract_raw_parts:
        raise RuntimeError(
            "No tract ACS data was fetched. Check CENSUS_API_KEY and target year range."
        )

    tract_panel_df = pd.concat(tract_raw_parts, ignore_index=True)
    print(f"Raw tract panel: {tract_panel_df.shape}")

    tract_acs_filled_df = fill_acs_panel(
        tract_panel_df,
        geoid_col=HT_INCOME_CONFIG["tract_key"],
        value_source_pairs=[
            ("median_income", "income_source"),
            ("population", "population_source"),
            ("households", "households_source"),
        ],
    )

    tract_acs_filled_df.to_csv(tract_ref_path, index=False)
    print("Saved:", tract_ref_path)
    print("Shape:", tract_acs_filled_df.shape)


Loaded cached tract ACS data: .\Inputs\tracts_2020_acs_income.csv
Shape: (3170, 8)


In [208]:
# Tract fill provenance spot check.
tract_fill_summary = (
    tract_acs_filled_df.groupby("year")
    .agg(
        total=("median_income", "count"),
        null_income=("median_income", lambda s: s.isna().sum()),
        income_acs=("income_source", lambda s: (s == "acs").sum()),
        income_interp=("income_source", lambda s: (s == "interpolated").sum()),
        income_extrap=("income_source", lambda s: (s == "extrapolated").sum()),
    )
    .reset_index()
)
print("Tract ACS fill summary:")
tract_fill_summary


Tract ACS fill summary:


,year,total,null_income,income_acs,income_interp,income_extrap
0,2019,452,3,455,0,0
1,2020,537,6,543,0,0
2,2021,537,6,543,0,0
3,2022,537,6,542,0,1
4,2023,537,6,542,0,1
5,2024,538,5,542,0,1


## 7. Fetch & Cache County-Level ACS Fallback

County-level ACS income is always available and robust (very large sample).  This section
provides the last-resort fallback for block groups whose income could not be filled from
the parent tract.  The same `fill_acs_panel` pipeline is applied and results are cached.

**File:** `.\Inputs\counties_2020_acs_income.csv`  
**Skipped:** if the file already exists.  Delete to force a full re-download.

In [209]:
county_ref_path = HT_INCOME_CONFIG["county_ref_path"]

if os.path.exists(county_ref_path):
    county_acs_filled_df = pd.read_csv(county_ref_path, dtype={HT_INCOME_CONFIG["county_key"]: str})
    print("Loaded cached county ACS data:", county_ref_path)
    print("Shape:", county_acs_filled_df.shape)
else:
    print("Fetching county-level ACS data for all target years...")
    county_raw_parts = []
    county_year_status = []

    for year in target_years:
        try:
            raw_df = get_census(
                dataset=HT_INCOME_CONFIG["acs_dataset"],
                variables=list(HT_INCOME_CONFIG["acs_vars"].values()),
                year=year,
                params={
                    "for": "county:*",
                    "in": f"state:{HT_INCOME_CONFIG['state_fips']}",
                    "key": CENSUS_API_KEY,
                },
                return_geoid=True,
                guess_dtypes=True,
            )
            # Derive county GEOID and filter to our 5 counties.
            raw_df[HT_INCOME_CONFIG["county_key"]] = (
                raw_df[HT_INCOME_CONFIG["bg_key"]].astype(str).str[:5]
            )
            raw_df = raw_df[
                raw_df[HT_INCOME_CONFIG["county_key"]].str[2:].isin(HT_INCOME_CONFIG["county_fips"])
            ].copy()

            raw_df = raw_df.rename(
                columns={
                    HT_INCOME_CONFIG["acs_vars"]["median_income"]: "median_income",
                    HT_INCOME_CONFIG["acs_vars"]["population"]: "population",
                    HT_INCOME_CONFIG["acs_vars"]["households"]: "households",
                }
            )
            for col in ("median_income", "population", "households"):
                raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")
                raw_df.loc[raw_df[col] <= 0, col] = np.nan

            raw_df["year"] = year
            raw_df["income_source"] = "acs"
            raw_df["population_source"] = "acs"
            raw_df["households_source"] = "acs"

            county_raw_parts.append(
                raw_df[
                    [
                        HT_INCOME_CONFIG["county_key"],
                        "median_income",
                        "population",
                        "households",
                        "year",
                        "income_source",
                        "population_source",
                        "households_source",
                    ]
                ]
            )
            county_year_status.append(
                {"year": year, "status": "fetched", "rows": len(raw_df), "message": "ok"}
            )
        except Exception as exc:
            county_year_status.append(
                {"year": year, "status": "missing", "rows": 0, "message": str(exc)}
            )

    print(pd.DataFrame(county_year_status))

    if not county_raw_parts:
        raise RuntimeError(
            "No county ACS data was fetched. Check CENSUS_API_KEY and target year range."
        )

    county_panel_df = pd.concat(county_raw_parts, ignore_index=True)
    print(f"Raw county panel: {county_panel_df.shape}")

    county_acs_filled_df = fill_acs_panel(
        county_panel_df,
        geoid_col=HT_INCOME_CONFIG["county_key"],
        value_source_pairs=[
            ("median_income", "income_source"),
            ("population", "population_source"),
            ("households", "households_source"),
        ],
    )

    county_acs_filled_df.to_csv(county_ref_path, index=False)
    print("Saved:", county_ref_path)
    print("Shape:", county_acs_filled_df.shape)


Loaded cached county ACS data: .\Inputs\counties_2020_acs_income.csv
Shape: (30, 8)


In [210]:
# County-level data should have no null income after fill.
print("County ACS filled data:")
county_acs_filled_df[
    [HT_INCOME_CONFIG["county_key"], "year", "median_income", "income_source"]
].sort_values([HT_INCOME_CONFIG["county_key"], "year"])


County ACS filled data:


,county_geoid,year,median_income,income_source
0,49003,2019,62233.0,acs
1,49003,2020,63573.0,acs
2,49003,2021,67486.0,acs
3,49003,2022,72769.0,acs
4,49003,2023,77865.0,acs
5,49003,2024,84550.0,acs
6,49011,2019,83310.0,acs
7,49011,2020,87570.0,acs
8,49011,2021,92765.0,acs
9,49011,2022,101285.0,acs


## 8. Apply Hierarchical Income Fallback (BG → Tract → County)

For every block group that still has a null `median_income` after Section 5 (whether due
to ACS suppression, API failure, or no observed years), this section fills the value from:

1. **Tract level** — the population-weighted mean income of the parent census tract
   (derived as the first 11 characters of the block group GEOID).
2. **County level** — the population-weighted mean income of the parent county
   (derived as the first 5 characters of the block group GEOID), used only when the
   tract value is also unavailable.

Only `median_income` and `income_source` are updated; `population` and `households`
remain as fetched by the ACS.  Cities with no block groups assigned (non-residential
TDM zones such as Utah Lake or Camp Williams) will still have null city-level income
after aggregation — this is correct behaviour and those cities receive `MEDINC = 0`
in the final export.

In [211]:
fallback_summary = []

for year in target_years:
    # ── Lookup tables: one income value per tract / county for this year ──────
    tract_lookup = (
        tract_acs_filled_df[tract_acs_filled_df["year"] == year][
            [HT_INCOME_CONFIG["tract_key"], "median_income"]
        ]
        .rename(columns={"median_income": "tract_income"})
        .drop_duplicates(HT_INCOME_CONFIG["tract_key"])
    )
    county_lookup = (
        county_acs_filled_df[county_acs_filled_df["year"] == year][
            [HT_INCOME_CONFIG["county_key"], "median_income"]
        ]
        .rename(columns={"median_income": "county_income"})
        .drop_duplicates(HT_INCOME_CONFIG["county_key"])
    )

    # ── Attach parent-geography keys to each BG row ───────────────────────────
    bg_df = bg_acs_by_year[year].copy()
    bg_df[HT_INCOME_CONFIG["bg_key"]] = bg_df[HT_INCOME_CONFIG["bg_key"]].astype(str)
    bg_df[HT_INCOME_CONFIG["tract_key"]] = bg_df[HT_INCOME_CONFIG["bg_key"]].str[:11]
    bg_df[HT_INCOME_CONFIG["county_key"]] = bg_df[HT_INCOME_CONFIG["bg_key"]].str[:5]

    bg_df = bg_df.merge(tract_lookup, on=HT_INCOME_CONFIG["tract_key"], how="left")
    bg_df = bg_df.merge(county_lookup, on=HT_INCOME_CONFIG["county_key"], how="left")

    n_before = int(bg_df["median_income"].isna().sum())

    # ── Tract fallback ─────────────────────────────────────────────────────────
    null_mask = bg_df["median_income"].isna()
    tract_avail = null_mask & bg_df["tract_income"].notna()
    bg_df.loc[tract_avail, "median_income"] = bg_df.loc[tract_avail, "tract_income"]
    bg_df.loc[tract_avail, "income_source"] = "tract_fallback"
    n_tract = int(tract_avail.sum())

    # ── County fallback ────────────────────────────────────────────────────────
    null_mask = bg_df["median_income"].isna()
    county_avail = null_mask & bg_df["county_income"].notna()
    bg_df.loc[county_avail, "median_income"] = bg_df.loc[county_avail, "county_income"]
    bg_df.loc[county_avail, "income_source"] = "county_fallback"
    n_county = int(county_avail.sum())

    n_after = int(bg_df["median_income"].isna().sum())
    fallback_summary.append(
        {
            "year": year,
            "null_before": n_before,
            "filled_by_tract": n_tract,
            "filled_by_county": n_county,
            "still_null": n_after,
        }
    )

    # ── Write updated income / source back; drop temporary join columns ────────
    update_df = bg_df[[HT_INCOME_CONFIG["bg_key"], "median_income", "income_source"]].copy()
    base_cols = [
        c for c in bg_acs_by_year[year].columns if c not in ("median_income", "income_source")
    ]
    bg_acs_by_year[year] = bg_acs_by_year[year][base_cols].merge(
        update_df, on=HT_INCOME_CONFIG["bg_key"], how="left"
    )

print("Hierarchical fallback complete.")
pd.DataFrame(fallback_summary)


Hierarchical fallback complete.


,year,null_before,filled_by_tract,filled_by_county,still_null
0,2019,21,4,17,0
1,2020,20,14,6,0
2,2021,20,14,6,0
3,2022,21,15,6,0
4,2023,21,15,6,0
5,2024,19,14,5,0


In [212]:
# Provenance breakdown after fallback: how many BGs got each income source?
source_summary = []
for year in target_years:
    counts = bg_acs_by_year[year]["income_source"].value_counts(dropna=False)
    row = {"year": year}
    for src in ("acs", "interpolated", "extrapolated", "tract_fallback", "county_fallback"):
        row[src] = int(counts.get(src, 0))
    row["still_null"] = int(bg_acs_by_year[year]["median_income"].isna().sum())
    source_summary.append(row)
pd.DataFrame(source_summary)


,year,acs,interpolated,extrapolated,tract_fallback,county_fallback,still_null
0,2019,1053,0,473,4,17,0
1,2020,1497,11,19,14,6,0
2,2021,1503,19,5,14,6,0
3,2022,1496,25,5,15,6,0
4,2023,1496,20,10,15,6,0
5,2024,1501,0,27,14,5,0


## 9. Assign Block Groups to Cities (Spatial Join)

Block groups are assigned to exactly one city using a centroid-based spatial join
(`HAVE_THEIR_CENTER_IN`).  This avoids double-counting from polygon-intersection joins.
Unmatched block groups are logged and excluded from aggregation.

The BG reference is in GCS NAD83 (EPSG:4269); the city shapefile is in UTM Zone 12N
(EPSG:26912).  A `CopyFeatures` + `Project` pattern re-projects the BGs — `CopyFeatures`
is required first to avoid ERROR 001489 (topology participation prevents direct `Project`).

In [213]:
city_fc = r".\Inputs\city_area_with_workshop_areas.shp"
bg_copy_fc = os.path.join(gdb, HT_INCOME_CONFIG["bg_projected_name"] + "_copy")
bg_projected_fc = os.path.join(gdb, HT_INCOME_CONFIG["bg_projected_name"])
bg_city_join_fc = os.path.join(gdb, HT_INCOME_CONFIG["bg_city_join_name"])
bg_join_input_fc = bg_reference_fc

bg_sr = arcpy.Describe(bg_reference_fc).spatialReference
city_sr = arcpy.Describe(city_fc).spatialReference

print("BG SR  :", bg_sr.name, bg_sr.factoryCode)
print("City SR:", city_sr.name, city_sr.factoryCode)

if bg_sr.factoryCode != city_sr.factoryCode:
    if arcpy.Exists(bg_copy_fc):
        arcpy.management.Delete(bg_copy_fc)
    arcpy.management.CopyFeatures(bg_reference_fc, bg_copy_fc)

    if arcpy.Exists(bg_projected_fc):
        arcpy.management.Delete(bg_projected_fc)
    arcpy.management.Project(
        in_dataset=bg_copy_fc, out_dataset=bg_projected_fc, out_coor_system=city_sr
    )
    arcpy.management.Delete(bg_copy_fc)

    bg_join_input_fc = bg_projected_fc
    print("Projected to:", city_sr.name)
else:
    print("CRS match — no projection needed")


BG SR  : GCS_North_American_1983 4269
City SR: NAD_1983_UTM_Zone_12N 26912
Projected to: NAD_1983_UTM_Zone_12N


In [214]:
if arcpy.Exists(bg_city_join_fc):
    arcpy.management.Delete(bg_city_join_fc)

arcpy.analysis.SpatialJoin(
    target_features=bg_join_input_fc,
    join_features=city_fc,
    out_feature_class=bg_city_join_fc,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="HAVE_THEIR_CENTER_IN",
)


<Result '.\\Outputs\\scratch.gdb\\bg_city_spatial_join'>

In [215]:
# Read back the join result; apply workshop-area renames to SUBAREA.
bg_city_lookup = pd.DataFrame.spatial.from_featureclass(bg_city_join_fc)[
    ["GEOID", "CITY_NAME", "SUBAREA", "CO_NAME"]
].copy()
bg_city_lookup.rename(columns={"CITY_NAME": HT_INCOME_CONFIG["city_key"]}, inplace=True)
bg_city_lookup[HT_INCOME_CONFIG["bg_key"]] = bg_city_lookup[HT_INCOME_CONFIG["bg_key"]].astype(str)
bg_city_lookup["CO_NAME"] = bg_city_lookup["CO_NAME"].str.upper()
bg_city_lookup["SUBAREA"] = bg_city_lookup["SUBAREA"].replace(
    HT_INCOME_CONFIG["workshop_area_lookup"]
)

bg_city_lookup.head()


,GEOID,CITYAREA,SUBAREA,CO_NAME
0,490351113061,Cottonwood Heights,Salt Lake County Se,SALT LAKE
1,490351113062,Cottonwood Heights,Salt Lake County Se,SALT LAKE
2,490351114001,South Salt Lake,Salt Lake County North,SALT LAKE
3,490351114002,South Salt Lake,Salt Lake County North,SALT LAKE
4,490351114003,South Salt Lake,Salt Lake County North,SALT LAKE


In [216]:
# Validate: duplicate GEOIDs indicate a join error and must be zero.
dup_count = bg_city_lookup[HT_INCOME_CONFIG["bg_key"]].duplicated().sum()
if dup_count > 0:
    raise ValueError(
        f"Duplicate GEOIDs in bg_city_lookup: {dup_count}. Investigate before continuing."
    )

print("Total BG rows    :", len(bg_city_lookup))
print("Duplicate GEOIDs :", dup_count)
print("Matched BGs      :", bg_city_lookup[HT_INCOME_CONFIG["city_key"]].notna().sum())
print(
    "Unmatched BGs    :",
    bg_city_lookup[HT_INCOME_CONFIG["city_key"]].isna().sum(),
    "(excluded from aggregation — expected for non-residential TDM zones)",
)

bg_city_lookup[
    [HT_INCOME_CONFIG["bg_key"], HT_INCOME_CONFIG["city_key"], "SUBAREA", "CO_NAME"]
].head(10)


Total BG rows    : 1547
Duplicate GEOIDs : 0
Matched BGs      : 1521
Unmatched BGs    : 26 (excluded from aggregation — expected for non-residential TDM zones)


,GEOID,CITYAREA,SUBAREA,CO_NAME
0,490351113061,Cottonwood Heights,Salt Lake County Se,SALT LAKE
1,490351113062,Cottonwood Heights,Salt Lake County Se,SALT LAKE
2,490351114001,South Salt Lake,Salt Lake County North,SALT LAKE
3,490351114002,South Salt Lake,Salt Lake County North,SALT LAKE
4,490351114003,South Salt Lake,Salt Lake County North,SALT LAKE
5,490351114005,South Salt Lake,Salt Lake County North,SALT LAKE
6,490351114006,South Salt Lake,Salt Lake County North,SALT LAKE
7,490572002022,Ogden,Weber County South,WEBER
8,490351117023,South Salt Lake,Salt Lake County North,SALT LAKE
9,490351117024,South Salt Lake,Salt Lake County North,SALT LAKE


## 10. Aggregate to City-Level Income

For each year, block-group income and household counts are aggregated to the city level:

```
MEDINC_YYYY = sum(median_income × population) / sum(population)
TOTHH_YYYY  = sum(households)                              [for all matched BGs]
TOTPOP_YYYY = sum(population)                              [for all matched BGs]
```

Block groups with missing income, missing population, or zero population are excluded
from the weighted income calculation.  `TOTHH` and `TOTPOP` sums include only those
same contributing block groups for internal consistency.

In [217]:
city_income_parts = []

for year in target_years:
    year_df = bg_acs_by_year[year].copy()
    year_df[HT_INCOME_CONFIG["bg_key"]] = year_df[HT_INCOME_CONFIG["bg_key"]].astype(str)
    year_df = year_df.merge(bg_city_lookup, on=HT_INCOME_CONFIG["bg_key"], how="left")

    # Keep only rows that can contribute to the weighted average.
    year_df = year_df[
        year_df[HT_INCOME_CONFIG["city_key"]].notna()
        & year_df["median_income"].notna()
        & year_df["population"].notna()
        & (year_df["population"] > 0)
    ].copy()

    year_df["weighted_income"] = year_df["median_income"] * year_df["population"]

    city_year = (
        year_df.groupby(HT_INCOME_CONFIG["city_key"], dropna=False)
        .agg(
            **{
                f"TOTPOP{year}": ("population", "sum"),
                f"TOTHH{year}": ("households", "sum"),
                f"MEDINC{year}": ("weighted_income", "sum"),
                f"bg_count_{year}": (HT_INCOME_CONFIG["bg_key"], "count"),
            }
        )
        .reset_index()
    )
    # Convert weighted sum → weighted mean.
    city_year[f"MEDINC{year}"] = (city_year[f"MEDINC{year}"] / city_year[f"TOTPOP{year}"]).round(0)

    city_income_parts.append(city_year)


In [218]:
# Merge all year slices into one wide DataFrame keyed on CITYAREA.
city_income_df = city_income_parts[0].copy()
for part in city_income_parts[1:]:
    city_income_df = city_income_df.merge(part, on=HT_INCOME_CONFIG["city_key"], how="outer")

print("Cities with any income data:", len(city_income_df))
print("Duplicate city rows:", city_income_df[HT_INCOME_CONFIG["city_key"]].duplicated().sum())
city_income_df.head(10)


Cities with any income data: 90
Duplicate city rows: 0


,CITYAREA,TOTPOP2019,TOTHH2019,MEDINC2019,bg_count_2019,TOTPOP2020,TOTHH2020,MEDINC2020,bg_count_2020,TOTPOP2021,TOTHH2021,MEDINC2021,bg_count_2021,TOTPOP2022,TOTHH2022,MEDINC2022,bg_count_2022,TOTPOP2023,TOTHH2023,MEDINC2023,bg_count_2023,TOTPOP2024,TOTHH2024,MEDINC2024,bg_count_2024
0,Alpine,9794.0,2567.0,135124.0,7.0,10208.0,2627.0,143417.0,7,9756.0,2552.0,147254.0,7,9484.0,2554.0,168423.0,7,9649.0,2697.0,159966.0,7,9566.0,2796.0,162409.0,7
1,American Fork,33624.0,9733.0,84222.0,22.0,33780.0,9799.0,86650.0,22,34725.0,10334.0,91186.0,22,36227.0,10841.0,100624.0,22,37988.0,11642.0,107877.0,22,39785.0,12326.0,111523.0,22
2,Balance of BOX ELDER,5295.0,1794.0,79931.0,4.0,5126.0,1705.0,85565.0,4,5328.0,1715.0,92167.0,4,5582.0,1778.0,103818.0,4,6109.0,1957.0,108261.0,4,6703.0,2065.0,116198.0,4
3,Bluffdale,14286.0,3677.0,99622.0,5.0,14802.0,3975.0,111512.0,5,16576.0,4596.0,107601.0,5,17460.0,5167.0,113575.0,5,18168.0,5483.0,126394.0,5,18797.0,5714.0,138150.0,5
4,Bountiful,43322.0,14324.0,88632.0,31.0,41527.0,13538.0,89462.0,31,43069.0,13782.0,96733.0,31,42676.0,13723.0,103396.0,31,42451.0,13747.0,105908.0,31,42334.0,13759.0,101885.0,31
5,Box Elder County North,1704.0,444.0,69722.0,1.0,1946.0,506.0,72083.0,1,1769.0,468.0,76346.0,1,1735.0,464.0,113611.0,1,1753.0,499.0,88250.0,1,1837.0,543.0,94554.0,1
6,Brigham City,15712.0,5638.0,50706.0,13.0,15886.0,5611.0,54135.0,13,15897.0,5603.0,57367.0,13,16264.0,5815.0,60549.0,13,16465.0,5803.0,66293.0,13,16625.0,5778.0,71591.0,13
7,Cedar Fort,NaN,NaN,NaN,NaN,386.0,122.0,61898.0,1,582.0,160.0,102632.0,1,959.0,264.0,103250.0,1,1184.0,336.0,124286.0,1,2466.0,712.0,119643.0,1
8,Cedar Hills,5932.0,1535.0,95009.0,4.0,6597.0,1643.0,111954.0,4,6284.0,1560.0,117943.0,4,6292.0,1581.0,128312.0,4,6422.0,1615.0,138708.0,4,6589.0,1710.0,136171.0,4
9,Centerville,17370.0,5755.0,93704.0,7.0,16420.0,5778.0,95754.0,7,16233.0,5706.0,104084.0,7,16316.0,5660.0,110059.0,7,16086.0,5617.0,117270.0,7,16120.0,5725.0,120122.0,7


In [219]:
# Aggregation spot check: manually verify weighted income and TOTHH for one city-year.
check_year = target_years[0]
check_city = city_income_df[HT_INCOME_CONFIG["city_key"]].dropna().iloc[0]

check_df = bg_acs_by_year[check_year].copy()
check_df[HT_INCOME_CONFIG["bg_key"]] = check_df[HT_INCOME_CONFIG["bg_key"]].astype(str)
check_df = check_df.merge(bg_city_lookup, on=HT_INCOME_CONFIG["bg_key"], how="left")
check_df = check_df[
    (check_df[HT_INCOME_CONFIG["city_key"]] == check_city)
    & check_df["median_income"].notna()
    & check_df["population"].notna()
    & (check_df["population"] > 0)
].copy()
check_df["weighted_income"] = check_df["median_income"] * check_df["population"]

computed_inc = round(check_df["weighted_income"].sum() / check_df["population"].sum(), 0)
computed_hh = check_df["households"].sum()
stored_inc = city_income_df.loc[
    city_income_df[HT_INCOME_CONFIG["city_key"]] == check_city, f"MEDINC{check_year}"
].iloc[0]
stored_hh = city_income_df.loc[
    city_income_df[HT_INCOME_CONFIG["city_key"]] == check_city, f"TOTHH{check_year}"
].iloc[0]

print(f"City: {check_city} | Year: {check_year}")
print(
    f"Manually computed MEDINC : {computed_inc}  |  Stored: {stored_inc}  |  Match: {computed_inc == stored_inc}"
)
print(f"Manually computed TOTHH  : {computed_hh}   |  Stored: {stored_hh}")


City: Alpine | Year: 2019
Manually computed MEDINC : 135124.0  |  Stored: 135124.0  |  Match: True
Manually computed TOTHH  : 2567.0   |  Stored: 2567.0


## 11. Fetch & Cache ACS Place Income / Build Hybrid MEDINC

For incorporated cities whose `CITYAREA` name matches a Census Place name exactly (~74
cities), ACS Place-level `B19013_001E` is more accurate than the BG-weighted estimate
because it is the directly reported Census figure rather than a weighted mean of
suppressed / interpolated block-group values.

**Source hierarchy applied to `MEDINC{YYYY}`:**

| City type | Source |
|---|---|
| Exact Census Place name match (~74 cities) | ACS Place B19013 |
| `Davis County` / `Weber County` whole-county TDM rows | ACS County (Section 7 CSV) |
| Sub-county / unincorporated / non-residential rows | BG-weighted (Section 10) |

`TOTPOP{YYYY}` and `TOTHH{YYYY}` always come from the BG-weighted aggregation.

**Cache file:** `.\Inputs\places_2020_acs_income.csv` — downloaded once, skipped on
re-runs.  Delete to force a full re-download (e.g. after adding a new target year).

In [ ]:
from pygris import places as fetch_places

places_ref_path = HT_INCOME_CONFIG["places_ref_path"]

if os.path.exists(places_ref_path):
    place_income_filled_df = pd.read_csv(places_ref_path)
    print("Loaded cached Place ACS income:", places_ref_path)
    print("Shape:", place_income_filled_df.shape)

else:
    print("Fetching ACS Place-level income for all target years...")

    # Fetch 2020 Census Places geometry for Utah.
    places_ut_gdf = fetch_places(
        state=HT_INCOME_CONFIG["state"], year=2020, cache=HT_INCOME_CONFIG["pygris_cache"]
    )

    # ── Build a deduplicated GEOID → NAME lookup ──────────────────────────────
    # Utah has place names that belong to more than one GEOID
    # (e.g. "Enterprise" is both an incorporated city and a CDP).
    # When a NAME appears multiple times, keep the incorporated place
    # (NAMELSAD containing " city" or " town") and drop the CDP.
    # This prevents duplicate (place_name, year) rows from breaking the pivot.
    places_ref = places_ut_gdf[["GEOID", "NAME", "NAMELSAD"]].copy()
    places_ref["is_incorporated"] = (
        places_ref["NAMELSAD"].str.lower().str.contains(r"\bcity\b|\btown\b", regex=True)
    )

    # Sort so incorporated rows come first, then deduplicate on NAME
    places_dedup = places_ref.sort_values("is_incorporated", ascending=False).drop_duplicates(
        subset="NAME", keep="first"
    )

    dupe_names = places_ref.groupby("NAME")["GEOID"].nunique().loc[lambda s: s > 1].index.tolist()
    if dupe_names:
        print(f"Duplicate place names resolved (kept incorporated): {dupe_names}")

    geoid_to_place_name = places_dedup.set_index("GEOID")["NAME"].to_dict()

    place_raw_parts = []
    place_year_status = []

    for year in target_years:
        try:
            raw = get_census(
                dataset=HT_INCOME_CONFIG["acs_dataset"],
                variables=["B19013_001E"],
                year=year,
                params={
                    "for": "place:*",
                    "in": f"state:{HT_INCOME_CONFIG['state_fips']}",
                    "key": CENSUS_API_KEY,
                },
                return_geoid=True,
                guess_dtypes=True,
            )
            raw = raw.rename(columns={"B19013_001E": "place_medinc"})
            raw["place_medinc"] = pd.to_numeric(raw["place_medinc"], errors="coerce")
            raw.loc[raw["place_medinc"] <= 0, "place_medinc"] = np.nan
            raw["place_name"] = raw["GEOID"].astype(str).map(geoid_to_place_name)
            raw["year"] = year
            raw["medinc_source"] = "acs_place"

            place_raw_parts.append(
                raw[["place_name", "year", "place_medinc", "medinc_source"]].dropna(
                    subset=["place_name"]
                )
            )
            place_year_status.append({"year": year, "status": "ok", "n": len(raw)})

        except Exception as exc:
            place_year_status.append({"year": year, "status": "error", "message": str(exc)})

    print(pd.DataFrame(place_year_status).to_string(index=False))

    if not place_raw_parts:
        raise RuntimeError(
            "No Place ACS data fetched — check CENSUS_API_KEY and target year range."
        )

    place_panel_df = pd.concat(place_raw_parts, ignore_index=True)

    # Final guard: drop any remaining duplicate (place_name, year) pairs
    dupes = place_panel_df.duplicated(subset=["place_name", "year"], keep=False)
    if dupes.any():
        print(f"WARNING: {dupes.sum()} duplicate (place_name, year) rows — keeping first.")
        place_panel_df = place_panel_df.drop_duplicates(subset=["place_name", "year"], keep="first")

    # Apply the same interp/extrap pipeline used for BG, tract, and county panels.
    place_income_filled_df = fill_acs_panel(
        place_panel_df,
        geoid_col="place_name",
        value_source_pairs=[("place_medinc", "medinc_source")],
    )

    place_income_filled_df.to_csv(places_ref_path, index=False)
    print("Saved:", places_ref_path)
    print("Shape:", place_income_filled_df.shape)


Fetching ACS Place-level income for all target years...
Using FIPS code '49' for input 'UT'
Duplicate place names resolved (kept incorporated): ['Enterprise']
 year status   n
 2019     ok 327
 2020     ok 333
 2021     ok 333
 2022     ok 333
 2023     ok 334
 2024     ok 334
Saved: .\Inputs\places_2020_acs_income.csv
Shape: (1986, 4)


In [ ]:
# TDM rows that correspond to a whole-county ACS figure.
# Sub-county rows (Box Elder North/South, SL County East Cyns variants, etc.)
# are intentionally excluded — they fall back to BG-weighted (Section 10).
COUNTY_TDM_ROWS = {"Davis County": "49011", "Weber County": "49057"}

# ── Pivot Place income to wide format ─────────────────────────────────────────
# pivot_table with aggfunc="mean" guards against any duplicate (place_name, year)
# pairs that might survive the dedup step above.
place_wide = place_income_filled_df.pivot_table(
    index="place_name", columns="year", values="place_medinc", aggfunc="mean"
)
place_wide.columns = [f"place_medinc_{yr}" for yr in place_wide.columns]
place_wide = place_wide.reset_index()

# ── Build county lookup from the already-cached county panel (Section 7) ──────
# county_acs_filled_df is already in memory — no new download needed.
county_wide = county_acs_filled_df.pivot(
    index=HT_INCOME_CONFIG["county_key"], columns="year", values="median_income"
)
county_wide.columns = [f"county_medinc_{yr}" for yr in county_wide.columns]
county_wide = county_wide.reset_index()

county_tdm_rows = []
for tdm_name, fips in COUNTY_TDM_ROWS.items():
    row = county_wide[county_wide[HT_INCOME_CONFIG["county_key"]] == fips]
    if len(row) == 1:
        entry = {"CITYAREA": tdm_name}
        for yr in target_years:
            col = f"county_medinc_{yr}"
            entry[col] = row[col].iloc[0] if col in row.columns else np.nan
        county_tdm_rows.append(entry)
    else:
        print(f"WARNING: county FIPS {fips} not found in county ACS for '{tdm_name}'")

county_tdm_df = pd.DataFrame(county_tdm_rows) if county_tdm_rows else pd.DataFrame()

# ── Start from BG-weighted MEDINC as the base ─────────────────────────────────
medinc_hybrid_df = city_income_df.copy()

# Join Place income
medinc_hybrid_df = medinc_hybrid_df.merge(
    place_wide, left_on="CITYAREA", right_on="place_name", how="left"
)

# Join county income for county TDM rows
if not county_tdm_df.empty:
    medinc_hybrid_df = medinc_hybrid_df.merge(county_tdm_df, on="CITYAREA", how="left")
else:
    for yr in target_years:
        medinc_hybrid_df[f"county_medinc_{yr}"] = np.nan

# ── Apply override hierarchy per year ─────────────────────────────────────────
# Priority: ACS Place > ACS County > BG-weighted (already in MEDINC{yr})
# medinc_source_records tracks which source each city resolved to — internal
# only, not exported (would require MEDINC_SRC{YYYY} wide columns).
medinc_source_records = []

for yr in target_years:
    place_col_yr = f"place_medinc_{yr}"
    county_col_yr = f"county_medinc_{yr}"
    medinc_col = f"MEDINC{yr}"

    for idx, row in medinc_hybrid_df.iterrows():
        place_val = row.get(place_col_yr, np.nan)
        county_val = row.get(county_col_yr, np.nan)
        bg_val = row[medinc_col]

        if pd.notna(place_val) and place_val > 0:
            medinc_hybrid_df.at[idx, medinc_col] = place_val
            src = "acs_place"
        elif pd.notna(county_val) and county_val > 0:
            medinc_hybrid_df.at[idx, medinc_col] = county_val
            src = "acs_county"
        else:
            src = "bg_weighted" if pd.notna(bg_val) and bg_val > 0 else "none"

        if yr == target_years[-1]:
            medinc_source_records.append({"CITYAREA": row["CITYAREA"], "medinc_source": src})

# Drop temporary join columns — not part of the export schema
temp_cols = (
    [f"place_medinc_{yr}" for yr in target_years]
    + [f"county_medinc_{yr}" for yr in target_years]
    + ["place_name"]
)
medinc_hybrid_df = medinc_hybrid_df.drop(
    columns=[c for c in temp_cols if c in medinc_hybrid_df.columns]
)

print("medinc_hybrid_df shape:", medinc_hybrid_df.shape)


medinc_hybrid_df shape: (90, 25)


In [222]:
# Internal source summary for most-recent year — not exported.
source_summary_df = (
    pd.DataFrame(medinc_source_records)
    .merge(medinc_hybrid_df[["CITYAREA", f"MEDINC{target_years[-1]}"]], on="CITYAREA", how="left")
    .rename(columns={f"MEDINC{target_years[-1]}": "MEDINC_final"})
    .sort_values("medinc_source")
)

counts = source_summary_df["medinc_source"].value_counts()
print(f"MEDINC source breakdown ({target_years[-1]}):")
print(f"  acs_place   : {counts.get('acs_place', 0):3d} cities")
print(f"  acs_county  : {counts.get('acs_county', 0):3d} cities")
print(f"  bg_weighted : {counts.get('bg_weighted', 0):3d} cities")
print(f"  none (NaN)  : {counts.get('none', 0):3d} cities")
print()
print(source_summary_df.to_string(index=False))


MEDINC source breakdown (2024):
  acs_place   :  74 cities
  acs_county  :   2 cities
  bg_weighted :  14 cities
  none (NaN)  :   0 cities

                   CITYAREA medinc_source  MEDINC_final
               Weber County    acs_county       90005.0
               Davis County    acs_county      110884.0
                     Alpine     acs_place      168929.0
             Salt Lake City     acs_place       75090.0
                      Salem     acs_place      111117.0
                        Roy     acs_place       91282.0
                   Riverton     acs_place      126910.0
                  Riverdale     acs_place       67323.0
                      Provo     acs_place       64171.0
              Pleasant View     acs_place      129462.0
             Pleasant Grove     acs_place      101073.0
                 Plain City     acs_place      132766.0
                      Perry     acs_place      112639.0
                     Payson     acs_place       89905.0
                   

## 12. Compute H+T Income Share and Build Export

Three new fields are computed for each city-year:

```
HTCOST{YYYY} = HCOST{YYYY} + TCOST{YYYY}                      (monthly $)
HPLUST{YYYY} = (HTCOST{YYYY} × 12) / MEDINC{YYYY}            (annual share, 0–1)
```

**fillna logic (split by column type):**
- `HCOST`, `TCOST`, `HTCOST` → filled with **0** (no TDM cost assigned is a real zero)
- `HPLUST` → stays **NaN** where MEDINC is null/zero (renders as N/A in the dashboard,
  not "0% of income spent on H+T")
- `MEDINC`, `TOTPOP`, `TOTHH` → filled with **0** (consistent, downstream can filter)

A guard drops previously derived columns before merging so this section is safe to re-run.

In [223]:
# Drop previously derived columns so re-runs do not accumulate duplicates.
derived_prefixes = ("MEDINC", "TOTPOP", "TOTHH", "bg_count_", "HTCOST")
existing_derived = [c for c in ht_df.columns if c.startswith(derived_prefixes)]
if existing_derived:
    ht_df = ht_df.drop(columns=existing_derived)

# Merge city-level ACS income / households into the H+T table.
ht_df = ht_df.merge(medinc_hybrid_df, on=HT_INCOME_CONFIG["city_key"], how="left")

# Compute monthly combined cost (HTCOST) and income share (HPLUST) for each year.
for year in target_years:
    htcost_col = f"HTCOST{year}"
    income_col = f"MEDINC{year}"
    ht_df[htcost_col] = np.where(
        ht_df[f"HCOST{year}"].notna() & ht_df[f"TCOST{year}"].notna(),
        ht_df[f"HCOST{year}"] + ht_df[f"TCOST{year}"],
        np.nan,
    )
    ht_df[f"HPLUST{year}"] = np.where(
        ht_df[income_col].notna() & (ht_df[income_col] > 0),
        (ht_df[htcost_col] * 12) / ht_df[income_col],
        np.nan,  # stays NaN — not 0 — for no-income cities
    )

# Summary.
share_cols = [f"HPLUST{year}" for year in target_years]
htcost_cols = [f"HTCOST{year}" for year in target_years]
income_cols = [f"MEDINC{year}" for year in target_years]

print(
    ht_df[
        [HT_INCOME_CONFIG["city_key"]] + income_cols[:2] + htcost_cols[:2] + share_cols[:2]
    ].head()
)
print()
for year in target_years:
    s = ht_df[f"HPLUST{year}"]
    print(
        f"{year}  non-null: {s.notna().sum():3d}"
        f"  null (no MEDINC): {s.isna().sum():3d}"
        f"  min: {s.min():.4f}  max: {s.max():.4f}"
    )


               CITYAREA  MEDINC2019  MEDINC2020  HTCOST2019  HTCOST2020  \
0                Alpine    129239.0    123450.0      2591.0      2466.0   
1                  Alta         NaN         NaN       504.0       336.0   
2         American Fork     77857.0     78690.0      1950.0      1952.0   
3  Balance of BOX ELDER     79931.0     85565.0      1235.0       914.0   
4              Benjamin         NaN         NaN       587.0       486.0   

   HPLUST2019  HPLUST2020  
0    0.240578    0.239708  
1         NaN         NaN  
2    0.300551    0.297674  
3    0.185410    0.128183  
4         NaN         NaN  

2019  non-null:  84  null (no MEDINC):  17  min: 0.0228  max: 0.4038
2020  non-null:  84  null (no MEDINC):  17  min: 0.0189  max: 0.4160
2021  non-null:  84  null (no MEDINC):  17  min: 0.0232  max: 0.4393
2022  non-null:  84  null (no MEDINC):  17  min: 0.0373  max: 0.4538
2023  non-null:  84  null (no MEDINC):  17  min: 0.0263  max: 0.4683
2024  non-null:  84  null (no MEDIN

In [224]:
# Spot check: manually verify one city-year end-to-end.
check_year = target_years[0]
check_city = ht_df[HT_INCOME_CONFIG["city_key"]].dropna().iloc[0]
row = ht_df.loc[ht_df[HT_INCOME_CONFIG["city_key"]] == check_city].iloc[0]

print(f"City          : {check_city}")
print(f"Year          : {check_year}")
print(f"HCOST (mo)    : {row[f'HCOST{check_year}']}")
print(f"TCOST (mo)    : {row[f'TCOST{check_year}']}")
print(f"HTCOST (mo)   : {row[f'HTCOST{check_year}']}  (= HCOST + TCOST)")
print(f"MEDINC        : {row[f'MEDINC{check_year}']}")
print(f"TOTHH         : {row[f'TOTHH{check_year}']}")
print(f"Income share  : {row[f'HPLUST{check_year}']:.4f}  (= HTCOST × 12 / MEDINC, expect 0-1)")


City          : Alpine
Year          : 2019
HCOST (mo)    : 1829.0
TCOST (mo)    : 762.0
HTCOST (mo)   : 2591.0  (= HCOST + TCOST)
MEDINC        : 129239.0
TOTHH         : 2567.0
Income share  : 0.2406  (= HTCOST × 12 / MEDINC, expect 0-1)


In [225]:
# Build the export DataFrame: city geometry + H+T data.
export_df = city_area_shp[[HT_INCOME_CONFIG["city_key"], "SUBAREA", "CO_NAME", "SHAPE"]].merge(
    ht_df, on=HT_INCOME_CONFIG["city_key"], how="left"
)

export_df["SUBAREA"] = export_df["SUBAREA"].replace(HT_INCOME_CONFIG["workshop_area_lookup"])

print("Pre-collapse rows:", len(export_df))
print("Pre-collapse cols:", len(export_df.columns))


Pre-collapse rows: 109
Pre-collapse cols: 52


In [226]:
# Drop intermediate columns not part of the final schema.
drop_prefixes = ("bg_count_",)
drop_cols = [c for c in export_df.columns if c.startswith(drop_prefixes)]
if drop_cols:
    export_df = export_df.drop(columns=drop_cols)

# ── fillna — split by column type ────────────────────────────────────────────
# HCOST, TCOST, HTCOST: fill NaN with 0 (no TDM cost = real zero).
cost_fill_cols = []
for year in target_years:
    cost_fill_cols.extend([f"HCOST{year}", f"TCOST{year}", f"HTCOST{year}"])
cost_fill_cols = [c for c in cost_fill_cols if c in export_df.columns]
export_df[cost_fill_cols] = export_df[cost_fill_cols].fillna(0)

# MEDINC, TOTPOP, TOTHH: fill NaN with 0 (non-residential / unmatched zones).
stat_fill_cols = [
    c for c in export_df.columns if any(c.startswith(p) for p in ("MEDINC", "TOTPOP", "TOTHH"))
]
export_df[stat_fill_cols] = export_df[stat_fill_cols].fillna(0)

# HPLUST: intentionally left as NaN where MEDINC was null/zero.
# The dashboard renders NaN as N/A rather than "0% of income spent on H+T".

# ── Column ordering ────────────────────────────────────────────────────────────
ordered_cols = [HT_INCOME_CONFIG["city_key"], "SUBAREA", "CO_NAME", "SHAPE"]
for prefix in ["HCOST", "TCOST", "HTCOST", "HPLUST", "MEDINC", "TOTPOP", "TOTHH"]:
    for year in target_years:
        col = f"{prefix}{year}"
        if col in export_df.columns:
            ordered_cols.append(col)
remaining_cols = [c for c in export_df.columns if c not in ordered_cols]
export_df = export_df[ordered_cols + remaining_cols]

print("Final export columns:")
print(export_df.columns.tolist())
export_df.head()


Final export columns:
['CITYAREA', 'SUBAREA', 'CO_NAME', 'SHAPE', 'HCOST2019', 'HCOST2020', 'HCOST2021', 'HCOST2022', 'HCOST2023', 'HCOST2024', 'TCOST2019', 'TCOST2020', 'TCOST2021', 'TCOST2022', 'TCOST2023', 'TCOST2024', 'HTCOST2019', 'HTCOST2020', 'HTCOST2021', 'HTCOST2022', 'HTCOST2023', 'HTCOST2024', 'HPLUST2019', 'HPLUST2020', 'HPLUST2021', 'HPLUST2022', 'HPLUST2023', 'HPLUST2024', 'MEDINC2019', 'MEDINC2020', 'MEDINC2021', 'MEDINC2022', 'MEDINC2023', 'MEDINC2024', 'TOTPOP2019', 'TOTPOP2020', 'TOTPOP2021', 'TOTPOP2022', 'TOTPOP2023', 'TOTPOP2024', 'TOTHH2019', 'TOTHH2020', 'TOTHH2021', 'TOTHH2022', 'TOTHH2023', 'TOTHH2024']


,CITYAREA,SUBAREA,CO_NAME,SHAPE,HCOST2019,HCOST2020,HCOST2021,HCOST2022,HCOST2023,HCOST2024,TCOST2019,TCOST2020,TCOST2021,TCOST2022,TCOST2023,TCOST2024,HTCOST2019,HTCOST2020,HTCOST2021,HTCOST2022,HTCOST2023,HTCOST2024,HPLUST2019,HPLUST2020,HPLUST2021,HPLUST2022,HPLUST2023,HPLUST2024,MEDINC2019,MEDINC2020,MEDINC2021,MEDINC2022,MEDINC2023,MEDINC2024,TOTPOP2019,TOTPOP2020,TOTPOP2021,TOTPOP2022,TOTPOP2023,TOTPOP2024,TOTHH2019,TOTHH2020,TOTHH2021,TOTHH2022,TOTHH2023,TOTHH2024
0,Alpine,Utah County North,UTAH,"{""rings"": [[[433081.68319999985, 4477344.8058]...",1829.0,1858.0,2074.0,2640.0,2867.0,4341.0,762.0,608.0,701.0,716.0,738.0,749.0,2591.0,2466.0,2775.0,3356.0,3605.0,5090.0,0.240578,0.239708,0.240541,0.249205,0.275917,0.361572,129239.0,123450.0,138438.0,161602.0,156786.0,168929.0,9794.0,10208.0,9756.0,9484.0,9649.0,9566.0,2567.0,2627.0,2552.0,2554.0,2697.0,2796.0
1,Alta,NA,SALT LAKE,"{""rings"": [[[449359.0999999996, 4492074], [449...",0.0,0.0,0.0,0.0,0.0,0.0,504.0,336.0,408.0,447.0,474.0,441.0,504.0,336.0,408.0,447.0,474.0,441.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,American Fork,Utah County North,UTAH,"{""rings"": [[[433815.22979999986, 4466632.21309...",1521.0,1600.0,1788.0,2213.0,2400.0,3366.0,429.0,352.0,394.0,406.0,418.0,426.0,1950.0,1952.0,2182.0,2619.0,2818.0,3792.0,0.300551,0.297674,0.316339,0.347309,0.352901,0.460203,77857.0,78690.0,82772.0,90490.0,95823.0,98878.0,33624.0,33780.0,34725.0,36227.0,37988.0,39785.0,9733.0,9799.0,10334.0,10841.0,11642.0,12326.0
3,Balance of BOX ELDER,NA,BOX ELDER,"{""rings"": [[[417479, 4578625.4], [417425.5, 45...",0.0,0.0,0.0,0.0,0.0,0.0,1235.0,914.0,1014.0,1140.0,1340.0,1141.0,1235.0,914.0,1014.0,1140.0,1340.0,1141.0,0.185410,0.128183,0.132021,0.131769,0.148530,0.117833,79931.0,85565.0,92167.0,103818.0,108261.0,116198.0,5295.0,5126.0,5328.0,5582.0,6109.0,6703.0,1794.0,1705.0,1715.0,1778.0,1957.0,2065.0
4,Benjamin,Utah County South,UTAH,"{""rings"": [[[438601.23319999967, 4437471.4559]...",0.0,0.0,0.0,0.0,0.0,0.0,587.0,486.0,550.0,551.0,558.0,536.0,587.0,486.0,550.0,551.0,558.0,536.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [227]:
export_fc = os.path.join(gdb2, "Affordability_Housing_Transportation_Costs")

if arcpy.Exists(export_fc):
    arcpy.management.Delete(export_fc)

export_df.spatial.to_featureclass(location=export_fc, sanitize_columns=False)
print("Exported:", export_fc)

export_df.drop(columns=["SHAPE"], errors="ignore").to_csv(
    os.path.join(outputs[0], "Affordability_Housing_Transportation_Costs.csv"), index=False
)

print("CSV exported:", os.path.join(outputs[0], "Affordability_Housing_Transportation_Costs.csv"))


Exported: .\Outputs\Affordability_Housing_Transportation_Costs.gdb\Affordability_Housing_Transportation_Costs
CSV exported: .\Outputs\Affordability_Housing_Transportation_Costs.csv


## 13. Validate Export

In [228]:
export_check = pd.DataFrame.spatial.from_featureclass(export_fc)

print("Export shape:", export_check.shape)
print(export_check.columns.tolist())

check_cols = [
    "CITYAREA",
    "SUBAREA",
    "CO_NAME",
    f"HCOST{target_years[0]}",
    f"TCOST{target_years[0]}",
    f"HTCOST{target_years[0]}",
    f"HPLUST{target_years[0]}",
    f"MEDINC{target_years[0]}",
    f"TOTPOP{target_years[0]}",
    f"TOTHH{target_years[0]}",
]
print(export_check[check_cols].head())


Export shape: (109, 47)
['OBJECTID', 'CITYAREA', 'SUBAREA', 'CO_NAME', 'HCOST2019', 'HCOST2020', 'HCOST2021', 'HCOST2022', 'HCOST2023', 'HCOST2024', 'TCOST2019', 'TCOST2020', 'TCOST2021', 'TCOST2022', 'TCOST2023', 'TCOST2024', 'HTCOST2019', 'HTCOST2020', 'HTCOST2021', 'HTCOST2022', 'HTCOST2023', 'HTCOST2024', 'HPLUST2019', 'HPLUST2020', 'HPLUST2021', 'HPLUST2022', 'HPLUST2023', 'HPLUST2024', 'MEDINC2019', 'MEDINC2020', 'MEDINC2021', 'MEDINC2022', 'MEDINC2023', 'MEDINC2024', 'TOTPOP2019', 'TOTPOP2020', 'TOTPOP2021', 'TOTPOP2022', 'TOTPOP2023', 'TOTPOP2024', 'TOTHH2019', 'TOTHH2020', 'TOTHH2021', 'TOTHH2022', 'TOTHH2023', 'TOTHH2024', 'SHAPE']
               CITYAREA            SUBAREA    CO_NAME  HCOST2019  TCOST2019  \
0                Alpine  Utah County North       UTAH       1829        762   
1                  Alta                 NA  SALT LAKE          0        504   
2         American Fork  Utah County North       UTAH       1521        429   
3  Balance of BOX ELDER           

In [229]:
# Per-year range check on the income share (stored in HPLUST columns).
# Expected: values between 0.0 and ~0.6; max above 1.0 would indicate a calculation error.
# NaN count expected for non-residential TDM zones (Utah Lake, Camp Williams, etc.).
print("Export feature class:", export_fc)
print("Rows   :", len(export_check))
print("Columns:", len(export_check.columns))
print()
for year in target_years:
    s = export_check[f"HPLUST{year}"]
    print(
        f"{year}  HPLUST  non-null: {s.notna().sum():3d}"
        f"  null (N/A): {s.isna().sum():3d}"
        f"  min: {s.min():.4f}  max: {s.max():.4f}"
    )


Export feature class: .\Outputs\Affordability_Housing_Transportation_Costs.gdb\Affordability_Housing_Transportation_Costs
Rows   : 109
Columns: 47

2019  HPLUST  non-null:  84  null (N/A):  25  min: 0.0228  max: 0.4038
2020  HPLUST  non-null:  84  null (N/A):  25  min: 0.0189  max: 0.4160
2021  HPLUST  non-null:  84  null (N/A):  25  min: 0.0232  max: 0.4393
2022  HPLUST  non-null:  84  null (N/A):  25  min: 0.0373  max: 0.4538
2023  HPLUST  non-null:  84  null (N/A):  25  min: 0.0263  max: 0.4683
2024  HPLUST  non-null:  84  null (N/A):  25  min: 0.0222  max: 0.5917


In [234]:
# Spot check: Alpine in the most recent year.
check_year = target_years[-1]
check_city = "Salt Lake City"

export_check.loc[
    export_check["CITYAREA"] == check_city,
    [
        "CITYAREA",
        f"HCOST{check_year}",
        f"TCOST{check_year}",
        f"HTCOST{check_year}",
        f"HPLUST{check_year}",
        f"MEDINC{check_year}",
        f"TOTHH{check_year}",
    ],
]


,CITYAREA,HCOST2024,TCOST2024,HTCOST2024,HPLUST2024,MEDINC2024,TOTHH2024
70,Salt Lake City,3376,187,3563,0.569397,75090,93367
